<a href="https://colab.research.google.com/github/embark-cybertraining/embark-scratch-notebooks/blob/main/obis_filter_by_areaid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Query OBIS for one species in one marine region and display summary information and map locations.

This notebook was constructed with the goal of porting functionality from an r script using "robis" OBIS_filter_by_areaid.R to python using "pyobis"

ChatGPT5.3 was utilized to help port the functionality with oversight, editing for purpose, and review by the embark project participants.

In [8]:
"""
Query OBIS for one species in one marine region and display summary information.

Purpose:
- Retrieve OBIS occurrence records for a scientific name within an areaid
- Display summary information
- Map locations of the occurrence records for data exploration

Usage:
- Edit SCIENTIFIC_NAME AREA_NAME, and AREA_ID as needed

"""

import requests
import folium
from datetime import datetime
import pandas as pd
from IPython.display import display

SCIENTIFIC_NAME = "Pyrosoma atlanticum"
AREA_ID = 40003
AREA_NAME = 'California Current'

def fetch_obis_occurrences(scientific_name, area_id, page_size=1000):
    """
    Fetch all OBIS occurrence records for one scientific name and one areaid.
    Returns a pandas DataFrame.
    """
    base_url = "https://api.obis.org/v3/occurrence"
    offset = 0
    all_results = []

    while True:
        params = {
            "scientificname": scientific_name,
            "areaid": area_id,
            "size": page_size,
            "from": offset,
        }

        response = requests.get(base_url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        results = payload.get("results", [])
        if not results:
            break

        all_results.extend(results)
        offset += page_size

        total = payload.get("total", 0)
        if offset >= total:
            break

    return pd.DataFrame(all_results)

records = fetch_obis_occurrences(SCIENTIFIC_NAME, AREA_ID)

## show summary info
print(f"Scientific name: {SCIENTIFIC_NAME}")
print(f"Area ID: {AREA_ID}")
print(f"Total records retrieved: {len(records)}")

Scientific name: Pyrosoma atlanticum
Area ID: 40003
Total records retrieved: 483


In [9]:
# If you want to save as a csv file:

output_file = "obis_occurrences_pyrosoma_atlanticum_california_current.csv"
records.to_csv(output_file, index=False)
print(f"Saved to: {output_file}")

Saved to: obis_occurrences_pyrosoma_atlanticum_california_current.csv


In [10]:
#More summary info

if records.empty:
    print("No records returned.")
else:
    print("\nColumns returned:")
    print(sorted(records.columns.tolist()))

    # Optional preview of key fields
    preview_cols = [
        "scientificName",
        "decimalLatitude",
        "decimalLongitude",
        "eventDate",
        "minimumDepthInMeters",
        "maximumDepthInMeters",
        "datasetID"
    ]

    print("\nPreview of key fields (random 10 lines)")
    display(records[preview_cols].sample(10))

    # Unique aphiaID values with WoRMS links

    aphia_values = pd.Series(records["aphiaID"].dropna().unique(), name="aphiaID")
    aphia_df = pd.DataFrame({"aphiaID": aphia_values})
    aphia_df["WoRMS_link"] = aphia_df["aphiaID"].apply(
        lambda x: f"https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}"
        if pd.notna(x) else None
    )

    display(aphia_df.sort_values("aphiaID").reset_index(drop=True))

    # Unique datasets, add OBIS links

    records["datasetURL"] = 'https://obis.org/dataset/'+ records['dataset_id'].astype(str)

    # Extract unique combinations
    datasets = (
        records[['dataset_id',"datasetName",'datasetID','datasetURL']]
        .drop_duplicates()
        .sort_values(by="datasetName")
        .reset_index(drop=True)
    )

    print(f"\nTotal unique datasets: {len(datasets)}")
    display(datasets)




Columns returned:
['absence', 'aphiaID', 'basisOfRecord', 'bathymetry', 'catalogNumber', 'class', 'classid', 'collectionCode', 'collectionID', 'continent', 'coordinateUncertaintyInMeters', 'country', 'county', 'datasetID', 'datasetName', 'dataset_id', 'dateIdentified', 'date_end', 'date_mid', 'date_start', 'date_year', 'day', 'decimalLatitude', 'decimalLongitude', 'depth', 'dropped', 'eventDate', 'eventID', 'eventRemarks', 'eventTime', 'family', 'familyid', 'fieldNumber', 'flags', 'genus', 'genusid', 'geodeticDatum', 'georeferenceProtocol', 'georeferenceRemarks', 'habitat', 'higherClassification', 'higherGeography', 'id', 'identificationRemarks', 'identifiedBy', 'individualCount', 'institutionCode', 'institutionID', 'kingdom', 'kingdomid', 'language', 'license', 'lifeStage', 'locality', 'locationID', 'locationRemarks', 'marine', 'materialSampleID', 'maximumDepthInMeters', 'maximumElevationInMeters', 'minimumDepthInMeters', 'minimumElevationInMeters', 'modified', 'month', 'node_id', 'n

,scientificName,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,datasetID
329,Pyrosoma atlanticum,37.274300,-122.815700,2014-05-05T06:49:36Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
192,Pyrosoma atlanticum,36.653300,-122.050000,1990-05-23T08:24:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
405,Pyrosoma atlanticum,36.296300,-122.091300,2012-06-12T07:45:09Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
286,Pyrosoma atlanticum,37.882800,-123.316300,2012-05-14T06:01:56Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
13,Pyrosoma atlanticum,45.466216,-124.755398,2023-04-15T18:17:29Z,553.30603,553.30603,noaa-oer-okeanos-ex2301
198,Pyrosoma atlanticum,36.755000,-121.982200,2014-05-28T09:16:51Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
238,Pyrosoma atlanticum,36.700000,-122.108300,1992-05-20T10:15:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
288,Pyrosoma atlanticum,37.272700,-122.568500,2014-05-20T04:33:32Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
91,Pyrosoma atlanticum,32.717000,-118.762500,2015-05-13T06:23:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
391,Pyrosoma atlanticum,36.986700,-122.591700,2001-05-14T08:25:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...


,aphiaID,WoRMS_link
0,137250,https://marinespecies.org/aphia.php?p=taxdetai...



Total unique datasets: 4


,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...
2,aa16d305-d413-4c4a-90be-b1ec3298d58d,NaN,NaN,https://obis.org/dataset/aa16d305-d413-4c4a-90...
3,c5687a17-e454-40f9-9a4b-d04b2c812d74,NaN,NaN,https://obis.org/dataset/c5687a17-e454-40f9-9a...


In [11]:
# Plot OBIS occurrence records from the `records` DataFrame

def plot_obis_records(records_df):
    """
    Return a folium map of OBIS occurrence points.
    """
    required = {"decimalLatitude", "decimalLongitude"}
    missing = required - set(records_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = records_df.dropna(subset=list(required))
    if df.empty:
        raise ValueError("No valid coordinate rows.")

    m = folium.Map(
        location=[df["decimalLatitude"].mean(), df["decimalLongitude"].mean()],
        zoom_start=4
    )

    #add this info in a popup using html table
    preview_cols = ["scientificName","decimalLatitude","decimalLongitude",
                    "eventDate","minimumDepthInMeters","maximumDepthInMeters" ]

    cols = [c for c in preview_cols if c in df.columns]

    for _, r in df.iterrows():
        rows = [
            f"<tr><th>{c}</th><td>{r[c]}</td></tr>"
            for c in cols if pd.notna(r.get(c))
        ]

        popup = folium.Popup(f"<table>{''.join(rows)}</table>", max_width=300)

        folium.CircleMarker(
            location=[r.decimalLatitude, r.decimalLongitude],
            radius=3,
            weight=1,
            fill=True,
            popup=popup,
        ).add_to(m)

    return m

obis_map = plot_obis_records(records)
obis_map

# Source Datasets and Metadata

Dataset info pattern
https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
* You can see "Overview" and "Data Quality" sections which include information about quality flags, missing data identifiers, coordinate precision, etc.

You can get the unique set of dataset_id column in records to look at metadata for any information you need about the source datasets in obis.

In [12]:
# We alread got the unique list of datasets from records
datasets

,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...
2,aa16d305-d413-4c4a-90be-b1ec3298d58d,NaN,NaN,https://obis.org/dataset/aa16d305-d413-4c4a-90...
3,c5687a17-e454-40f9-9a4b-d04b2c812d74,NaN,NaN,https://obis.org/dataset/c5687a17-e454-40f9-9a...


In [13]:
#You can visit the links for more info about the dataset
datasets["datasetURL"].dropna().tolist()


['https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f',
 'https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb',
 'https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d',
 'https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74']

In [14]:
# Dataset citation for OBIS
# * Citation based on https://manual.obis.org/citing.html

#citaion builder based on this script's variables
now = datetime.now()
year = now.year
access_date = now.strftime("%B %d, %Y")

citation = (
    f"Ocean Biodiversity Information System (OBIS). ({year}). "
    f"Occurrence records of {SCIENTIFIC_NAME} in the {AREA_NAME} (LME {AREA_ID}). "
    f"https://obis.org (accessed {access_date})."
)

print("OBIS citation:\n")
print(citation)

OBIS citation:

Ocean Biodiversity Information System (OBIS). (2026). Occurrence records of Pyrosoma atlanticum in the California Current (LME 40003). https://obis.org (accessed May 01, 2026).
